In [1]:
import os
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from pathlib import Path
import cv2
import pickle
import numpy as np
import matplotlib.pyplot as plt
import math
import json
from tqdm import tqdm
from datetime import date
plt.rcParams["figure.figsize"] = (24,18)

from ultralytics import YOLO
from cv_utils import *

In [2]:
# YOLO model path
model_path = os.path.join(os.path.dirname(os.getcwd()), 'models')

# General data path
data_path = os.path.join(os.path.dirname(os.getcwd()), 'data')

#### Crop images to football pitch

In [ ]:
all_cropped_destination = []
for vid_name in ['Liv-BHA 1st half tracking', 'MC-ARS 1st half tracking', 'MU-NU 2nd half tracking']:
    print(vid_name)
    image_root_path = os.path.join(data_path, 'images/' + vid_name)
    destination_path = os.path.join(data_path, 'images/cropped_' + vid_name)
    if not os.path.exists(destination_path):
        os.mkdir(destination_path)
    for image_name in tqdm(os.listdir(image_root_path)):
        image_path = os.path.join(image_root_path, image_name)
        image = cv2.imread(image_path, cv2.COLOR_BGR2RGB)

        # Crop the image to only the pitch
        cropped_image = pitch_segment(image)

        # Write image to destination
        destination_image_path = os.path.join(destination_path, image_name)
        cv2.imwrite(destination_image_path, cropped_image)
    all_cropped_destination.append(destination_path)

#### Generate COCO dataset

In [ ]:
# Generate one or multiple COCO datasets
destination_path_list = []
for folder_path in all_cropped_destination:
    vid_name = os.path.basename(folder_path)
    print(vid_name)
    dest_coco_path = export_coco_dataset_from_prediction(data_path, vid_name, 
                                        model_path=model_path, model_name="yolov8n_2nd_train.pt")
    
    destination_path_list.append(dest_coco_path)

#### Merge multiple COCO datasets

In [3]:
coco_dataset_path_1 = os.path.join(data_path, 'coco_datasets/real_train_pitch')
coco_dataset_path_2 = os.path.join(data_path, 'coco_datasets/synth_train_pitch')

dest_path=os.path.join(data_path, 'coco_datasets')

# Merge generated COCO dataset to one dataset for model training
merge_dest_path = merge_coco_dataset(coco_dataset_path_1, coco_dataset_path_2,
                   dest_path=os.path.join(data_path, 'coco_datasets'))

#### Convert COCO dataset to YOLO format

In [ ]:
# Load COCO dataset
dataset_name = 'merge_train_fixed_v2'
coco_dataset_path = os.path.join(data_path, 'coco_datasets/' + dataset_name)
test_dataset_path = os.path.join(data_path, 'coco_datasets/' + 'real_test_fixed')
# Convert COCO to YOLO
coco2yolo(coco_dataset_path, test_dataset_path=test_dataset_path)

  0%|          | 0/316 [00:00<?, ?it/s]

100%|██████████| 300/300 [00:13<00:00, 22.21it/s]


#### Augment YOLO dataset

In [ ]:
dataset_name = 'merge_train_fixed_v2_yolov8'
yolo_dataset_path = os.path.join(data_path, 'coco_datasets/' + dataset_name)
yolo_metadata_path = os.path.join(yolo_dataset_path, 'data.yaml')

augment_yolo(yolo_metadata_path, yolo_dataset_path)

/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/coco_datasets/merge_train_fixed_v2_yolov8/train/images


100%|██████████| 316/316 [03:52<00:00,  1.36it/s]

2844 2844
